# Leaf water potential
 

In [ ]:
# | default_exp leaf_water_potential

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
from plant_hydraulics.parameter_classes import PhysCon, Leaf, Flux

```
Input: ψ_soil (soil water supply)
       E (transpiration rate from energy balance)
       LSC (soil-to-leaf conductance from soil_resistance)
       h (leaf height)
       C (plant capacitance)
       ψ_leaf_old (current leaf water potential)
       dt (time step)
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Gravitational head                  │
    │  head = ρ_water · g  (MPa/m)         │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Steady-state target                 │
    │  a = ψ_soil − head·h − E/LSC         │
    │      ↑          ↑         ↑          │
    │    supply    gravity   transpiration │
    │    from      penalty   drawdown      │
    │    soil                              │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Time constant                       │
    │  τ = C / LSC  (seconds)              │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Analytical integration              │
    │  Δψ = (a − ψ₀) · (1 − e^(−dt/τ))     │
    │  ψ_leaf_new = ψ₀ + Δψ                │
    └──────────────────────────────────────┘
                    │
                    ▼
            Return ψ_leaf_new
                    │
          (used by stomatal_efficiency
           to check: ψ_leaf > ψ_min ?)
```

In [ ]:
# | export


def _k_weibull(psi, k_max, b, c):
    """
    Hydraulic conductance at water potential psi [MPa].

    Uses Weibull (exponential) vulnerability curve as in Sperry 2017:
        k(ψ) = k_max · exp(−(|ψ|/b)^c)

    Parameters
    ----------
    psi   : float  — water potential (MPa, negative)
    k_max : float  — maximum conductance (mmol/m²/s/MPa)
    b     : float  — Weibull scale parameter (MPa, positive)
    c     : float  — Weibull shape parameter (dimensionless, > 1)

    Returns
    -------
    float — conductance at psi (mmol/m²/s/MPa)
    """
    return k_max * np.exp(-((abs(psi) / b) ** c))

In [ ]:
# | export


def _E_supply(psi_leaf, psi_soil, k_max, b, c):
    """
    Hydraulic supply function: maximum transpiration rate supportable
    when leaf water potential is psi_leaf (Sperry 2017 Eq. 3).

    E(ψ_leaf) = ∫[ψ_leaf → ψ_soil] k_max · f(ψ) dψ

    The integral goes from psi_leaf (more negative) to psi_soil (less
    negative), giving a positive result.

    Parameters
    ----------
    psi_leaf : float — leaf water potential (MPa, negative)
    psi_soil : float — soil water potential (MPa, negative)
    k_max    : float — maximum conductance (mmol/m²/s/MPa)
    b, c     : float — Weibull parameters

    Returns
    -------
    float — supply-limited transpiration (mmol/m²/s)
    """
    if psi_leaf >= psi_soil:
        return 0.0
    val, _ = quad(lambda p: _k_weibull(p, k_max, b, c), psi_leaf, psi_soil)
    return val

In [ ]:
# | export


def _E_crit(psi_soil, k_max, b, c, psi_min=-30.0):
    """
    Critical (maximum possible) transpiration rate.
    Corresponds to ψ_leaf → −∞ (complete cavitation).

    Parameters
    ----------
    psi_soil : float — soil water potential (MPa)
    k_max    : float — maximum conductance (mmol/m²/s/MPa)
    b, c     : float — Weibull parameters
    psi_min  : float — numerical lower limit (−30 MPa is sufficient)

    Returns
    -------
    float — E_crit (mmol/m²/s)
    """
    val, _ = quad(lambda p: _k_weibull(p, k_max, b, c), psi_min, psi_soil)
    return val

In [ ]:
# | export


def _psi_from_supply(e_mmol, psi_soil, k_max, b, c, minl_wp):
    """
    Invert the supply function: find ψ_leaf such that E_supply(ψ_leaf) = e_mmol.
    Returns the EQUILIBRIUM leaf water potential for a given transpiration rate.
    """
    Ec = _E_crit(psi_soil, k_max, b, c)

    if e_mmol <= 0.0:
        return psi_soil
    if e_mmol >= Ec:
        return minl_wp  # demand exceeds supply capacity

    def residual(psi_l):
        return _E_supply(psi_l, psi_soil, k_max, b, c) - e_mmol

    try:
        return brentq(
            residual,
            minl_wp - 5.0,  # lower bound
            psi_soil - 1e-6,  # upper bound
            xtol=1e-5,
            maxiter=50,
        )
    except ValueError:
        # Numerical fallback: Ohm's law
        return psi_soil - e_mmol / k_max

In [ ]:
# | hide

# Old implementation!!!

# def leaf_water_potential(physcon: PhysCon, leaf: Leaf, flux: Flux) -> float:
#    """
#    Calculate leaf water potential for the current transpiration rate.
#
#    Solves the plant capacitance equation for the change in leaf water
#    potential over a model time step. The leaf water potential is driven
#    toward a steady-state value set by the balance between soil water
#    supply and transpiration demand, with the rate of approach governed
#    by the plant capacitance.
#
#    The steady-state target is:
#
#        a = ψ_soil - ρ·g·h - E / LSC
#
#    where the three terms represent soil water potential (i.e Pre-dawn), the
#    gravitational head (potential energy by unit of weight) for lifting water to leaf height, and the
#    transpiration-driven drawdown through the soil-to-leaf pathway.
#
#    The change in ψ_leaf is integrated analytically as:
#
#        Δψ = (a - ψ_leaf) · (1 - exp(-dt / τ))
#
#    where τ = C / LSC is the time constant (plant capacitance divided
#    by leaf-specific conductance).
#
#    The governing ODE is dy/dt = (a - y) / τ, where y = ψ_leaf,
#    a is the steady-state target, and τ = C / LSC. This is integrated
#    exactly over the time step dt to give:
#
#        dy = (a - y) · (1 - exp(-dt / τ))
#
#    The factor 1000 in the transpiration term converts flux.etflx
#    from mol H2O/m2/s to mmol H2O/m2/s for consistency with LSC
#    units.
#
#    The behaviour:
#
#    - If dt ≪ τ (short time step relative to the time constant):
#      the exponential ≈ 1 − dt/τ, so Δψ ≈ (a − ψ₀) · dt/τ — a small linear step
#      toward the target
#
#    - If dt ≫ τ (long time step): the exponential → 0, so Δψ ≈ a − ψ₀ — the leaf
#      jumps all the way to the steady state
#
#    - If ψ₀ = a already (leaf is at equilibrium): Δψ = 0 — nothing changes
#
#    __The bathtub analogy applied to dy = (a - y) · (1 - exp(-dt / τ))__
#
#    A leaf can be viewed as a bathtub where the inflow represents the water
#    supply from the soil and the outflow represents the transpiration.
#    In this analogy the water potential is the bathtub water level. In the
#    absence of capacitance, what comes in immediately comes out (Eq 13.6).
#    However, to represent the effect of capacitance over the water potential,
#    a tau_b term is incorporated (Eq 13.11). tau_b is the ratio between bathtub
#    size and pipe water supply (capacitance / leaf-specific conductance) and
#    represents the time it takes the water level to change. Therefore, if the
#    bathtub is big (large capacitance) and the water supply pipe is small,
#    then the ratio will be large, indicating that it takes a lot of time to
#    change the water level (water potential).
#
#    In other words tau_b it is the ratio that determines the respose time for the
#    change in water potential
#
#    __Parameters:__
#
#    - PhysCon: Physical constants:
#
#        - grav: Gravitational acceleration (m/s2).
#        - denh2o: Density of liquid water (kg/m3).
#
#    - Leaf: Leaf parameters:
#
#        - capac: Plant capacitance (mmol H2O/m2 leaf area/MPa).
#
#    - Flux: Flux variables:
#
#        - height: Leaf height (m).
#        - lsc: Leaf-specific conductance (mmol H2O/m2 leaf/s/MPa).
#        - psi_leaf: Leaf water potential at beginning of time step (MPa).
#        - psi_soil: Weighted soil water potential (MPa).
#        - etflx: Leaf transpiration flux (mol H2O/m2 leaf/s).
#        - dt: Model time step (s).
#
#    __Returns:__
#
#    leaf_wp: Leaf water potential at end of time step (MPa).
#
#    Parameters
#    ----------
#    physcon : PhysCon
#        Physical constants:
#        - grav : float
#            Gravitational acceleration (m/s2).
#        - denh2o : float
#            Density of liquid water (kg/m3).
#    leaf : Leaf
#        Leaf parameters:
#        - capac : float
#            Plant capacitance (mmol H2O/m2 leaf area/MPa).
#    flux : Flux
#        Flux variables:
#        - height : float
#            Leaf height (m).
#        - lsc : float
#            Leaf-specific conductance (mmol H2O/m2 leaf/s/MPa).
#        - psi_leaf : float
#            Leaf water potential at beginning of time step (MPa).
#        - psi_soil : float
#            Weighted soil water potential (MPa).
#        - etflx : float
#            Leaf transpiration flux (mol H2O/m2 leaf/s).
#        - dt : float
#            Model time step (s).
#
#    Returns
#    -------
#    leaf_wp : float
#        Leaf water potential at end of time step (MPa).
#    """
#
#    # Head of pressure (MPa/m)
#    # Add 0.01 MPa per meter of height
#    # potential energy by unit of weight
#    head = physcon.denh2o * physcon.grav * 1e-06
#
#    # Change in leaf water potential: dy/dt = (a - y) / b
#    # Integrated: dy = (a - y0) * (1 - exp(-dt/b))
#    y0 = flux.psi_leaf
#
#    # Steady state model --------------------------------------------------------
#
#    # (flux.etflx / flux.lsc)
#    # This is Ohm's law for water flow. The transpiration rate flux.etflx
#    # mol H₂O/m²/s) is the "current," and flux.lsc is the leaf-specific
#    # conductance — how easily water flows through the entire soil-to-leaf
#    # pathway (the "pipe width"). The factor 1000 converts mol to mmol for unit
#    # consistency with LSC.
#
#    # target_a represents what psi_leaf would be if the system reached
#    # equilibrium (meaning inflow = outflow therefore steady state)
#    # soil supply, minus the gravitational cost, minus the transpiration drawdown.
#    # head = potential energy by unit of weight
#    target_a = flux.psi_soil - head * flux.height - 1000.0 * flux.etflx / flux.lsc
#
#    # Incorporate Capacitance ---------------------------------------------------
#
#    # Capacitance can be thought of as the amount of water that can be extracted
#    # per unit change in water potential.
#    # leaf.capac (mol H2O / m2 /Pa)
#    # flux.lsc   (mmol H₂O / m2 leaf / s / MPa)
#
#    # tau_b represents how big is the water reservoir relative to the intake
#    # tau_b = time (seconds).
#
#    # A large capacitance (big spongy trunk that stores lots of water) means
#    # large tau_b
#
#    # A small capacitance (thin herbaceous stem with no storage) means small
#    # tau_b, and the leaf tracks the steady-state target almost instantaneously.
#
#    # tau_b it is the ratio that determines the respose time for the
#    # change in water potential
#    tau_b = leaf.capac / flux.lsc
#
#    # Eq 13.11 ------------------------------------------------------------------
#    dy = (target_a - y0) * (1.0 - np.exp(-flux.dt / tau_b))
#    leaf_wp = y0 + dy
#
#    return leaf_wp

In [ ]:
# | export


def leaf_water_potential(physcon, leaf, flux):
    """
    Leaf water potential with supply-curve equilibrium + capacitance dynamics.

    The computation has two independent stages:

    - Stage 1: Equilibrium target (ψ_eq):

        Where ψ_leaf WOULD settle if given infinite time.

        - Mode A (supply curve, when leaf.vc_b and leaf.vc_c are set):
            Invert  E = ∫[ψ_leaf→ψ_soil] k_max · f(ψ) dψ
            Correctly accounts for nonlinear conductance loss (cavitation).
            Faithful to Sperry 2017.

        - Mode B (Ohm's law fallback, original behaviour):
            ψ_eq = ψ_soil − gravity − E / lsc
            Assumes constant conductance. Used by Medlyn / WUE models
            which don't set vc_b / vc_c.

    - Stage 2 — Capacitance dynamics (Bonan Eq. 13.11):
        ψ_leaf moves toward ψ_eq exponentially:
            τ      = leaf.capac / flux.lsc          [time constant, s]
            dy     = (ψ_eq − ψ₀) × (1 − e^{−dt/τ})
            ψ_leaf = ψ₀ + dy

        When dt >> τ:  ψ_leaf → ψ_eq  (quasi-steady-state, Sperry 2017)
        When dt << τ:  ψ_leaf ≈ ψ₀    (capacitance buffers the change)

        Solves the plant capacitance equation for the change in leaf water
        potential over a model time step. The leaf water potential is driven
        toward a steady-state value set by the balance between soil water
        supply and transpiration demand, with the rate of approach governed
        by the plant capacitance.

        The steady-state target is:

            a = ψ_soil - ρ·g·h - E / LSC

        where the three terms represent soil water potential (i.e Pre-dawn), the
        gravitational head (potential energy by unit of weight) for lifting water
        to leaf height, and the transpiration-driven drawdown through the
        soil-to-leaf pathway.

        The change in ψ_leaf is integrated analytically as:

            Δψ = (a - ψ_leaf) · (1 - exp(-dt / τ))

        where τ = C / LSC is the time constant (plant capacitance divided
        by leaf-specific conductance).

        The governing ODE is dy/dt = (a - y) / τ, where y = ψ_leaf,
        a is the steady-state target, and τ = C / LSC. This is integrated
        exactly over the time step dt to give:

            dy = (a - y) · (1 - exp(-dt / τ))

        The factor 1000 in the transpiration term converts flux.etflx
        from mol H2O/m2/s to mmol H2O/m2/s for consistency with LSC
        units.

        The behaviour:

        - If dt ≪ τ (short time step relative to the time constant):
          the exponential ≈ 1 − dt/τ, so Δψ ≈ (a − ψ₀) · dt/τ — a small linear step
          toward the target

        - If dt ≫ τ (long time step): the exponential → 0, so Δψ ≈ a − ψ₀ — the leaf
          jumps all the way to the steady state

        - If ψ₀ = a already (leaf is at equilibrium): Δψ = 0 — nothing changes

        __The bathtub analogy applied to dy = (a - y) · (1 - exp(-dt / τ))__

        A leaf can be viewed as a bathtub where the inflow represents the water
        supply from the soil and the outflow represents the transpiration.
        In this analogy the water potential is the bathtub water level. In the
        absence of capacitance, what comes in immediately comes out (Eq 13.6).
        However, to represent the effect of capacitance over the water potential,
        a tau_b term is incorporated (Eq 13.11). tau_b is the ratio between bathtub
        size and pipe water supply (capacitance / leaf-specific conductance) and
        represents the time it takes the water level to change. Therefore, if the
        bathtub is big (large capacitance) and the water supply pipe is small,
        then the ratio will be large, indicating that it takes a lot of time to
        change the water level (water potential).

        In other words tau_b it is the ratio that determines the respose time for the
        change in water potential


    Parameters
    ----------
    physcon : PhysCon
        Physical constants (denh2o, grav).
    leaf    : Leaf
        Must always have:  capac, minl_wp
        For Mode A also:   vc_b, vc_c, gplant
        For Mode B also:   (uses flux.lsc, no extra leaf params)
    flux    : Flux
        etflx   — transpiration [mol H₂O/m²/s]
        psi_leaf — water potential at previous timestep [MPa]
        psi_soil — soil water potential [MPa]
        lsc      — leaf-specific conductance [mmol/m²/s/MPa]
        height   — canopy height [m]
        dt       — timestep [s]

    Returns
    -------
    flux with psi_leaf updated.
    """

    # Transpiration demand ------------------------------------------------------
    e_mmol = flux.etflx * 1000.0  # mol/m²/s → mmol/m²/s
    
    # ψ_leaf from previous timestep. Used as inital water potential
    y0 = flux.psi_leaf 
    
    # Stage 1: compute equilibrium target ψ_eq ----------------------------------
    use_supply_curve = (
        hasattr(leaf, "vc_b")
        and hasattr(leaf, "vc_c")
        and leaf.vc_b > 0
        and leaf.vc_c > 0
    )

    # Mode A: supply-curve inversion (Sperry 2017) ------------------------------
    if use_supply_curve:
        
        # ψ_eq = ψ_leaf that would result in E_supply = e_mmol at steady state
        psi_leaf = _psi_from_supply(
            e_mmol,
            flux.psi_soil,
            
            # k_max [mmol/m²/s/MPa]
            leaf.gplant,
            leaf.vc_b,
            leaf.vc_c,
            leaf.minl_wp,
        )

    # Mode B: Ohm's law (Bonan WUE / Medlyn) ------------------------------------
    else:
        
        # Gravitational head: ρgh converted to MPa (0.01 MPa per 1 m height)
        # Head of pressure (MPa/m)
        # head == potential energy by unit of weight
        # Add 0.01 MPa per meter of height  potential energy by unit of weight
        head = physcon.denh2o * physcon.grav * 1e-06

        # Steady state water potential calculation

        # Term (flux.etflx / flux.lsc)0
        # This is Ohm's law for water flow. The transpiration rate flux.etflx
        # mol H₂O/m²/s) is the "current," and flux.lsc is the leaf-specific
        # conductance, how easily water flows through the entire soil-to-leaf
        # pathway (the "pipe width"). 

        # soil supply minus the energy cost of lifting water 
        # (the taller the tree the higher the cost) minus the transpiration 
        # drawdown.
        
        # Term psi_eq represents what psi_leaf would be if the system reached
        # equilibrium (meaning inflow = outflow therefore steady state). In other
        # words psi_eq represents the wp at which the leaf would settle at a 
        # given infinite timeat the current transpiration rate 
        psi_eq = flux.psi_soil - head * flux.height - 1000.0 * flux.etflx / flux.lsc

        # Incorporate capacitance dynamics (Bonan Eq. 13.11) 

        # Capacitance can be thought of as the amount of water that can be 
        # extracted per unit change in water potential.

        # TODO: Check units of capaciatance and flux.lsc in book
        # leaf.capac (mmol H2O / m2 /MPa)
        # flux.lsc   (mmol H₂O / m2 leaf / s / MPa)

        # tau_b represents how big is the water reservoir relative to the intake
        # tau_b = time (seconds).

        # A large capacitance (big spongy trunk that stores lots of water) means
        # large tau_b

        # A small capacitance means small tau_b, and the leaf tracks the 
        # steady-state target almost instantaneously.

        # tau_b it is the ratio that determines the respose time for the
        # change in water potential

        # τ = C / k_eff  [seconds]
        
        # Use flux.lsc as the effective conductance for the RC time constant.
        tau_b = leaf.capac / flux.lsc

        # Eq 13.11 same as Eq 4.24.
        # dy is the intengral of the ODE dψ/dt = (psi_eq − ψ) / τ
        
        # dψ/dt = (psi_eq − ψ) / τ 
        #           ↓
        #      Integrates to:
        #           ↓
        # ψ(t) = psi_eq + (y0 − psi_eq) × exp(−dt/τ)
        #           ↓
        #     Which rearranges to:
        #           ↓
        #  ψ(t) − y0 = (psi_eq − y0) × (1 − exp(−dt/τ))
        
        # the right hand side is dy
        
        # dy represents how much does the wp move torward the target psi_eq at 
        # this time step
        dy = (psi_eq - y0) * (1.0 - np.exp(-flux.dt / tau_b))
        
        # Actuall water potential calculation
        psi_leaf = y0 + dy

    # print(f"[DEBUG] psi_leaf={psi_leaf}")
    # Hard safety floor ---------------------------------------------------------
    flux.psi_leaf = max(psi_leaf, leaf.minl_wp)
    return flux

#### Example leaf_water_potential()